In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

d:\ProteinPrediction\MainFiles\GPU_VENV\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
import numpy as np
import torch

# load numpy arrays
X_train_w = np.load("../data/Saved_Windows_features/X_train.npy")
y_train_w = np.load("../data/Saved_Windows_features/y_train.npy")

X_val_w = np.load("../data/Saved_Windows_features/X_val.npy")
y_val_w = np.load("../data/Saved_Windows_features/y_val.npy")

X_test_w = np.load("../data/Saved_Windows_features/X_test.npy")
y_test_w = np.load("../data/Saved_Windows_features/y_test.npy")

In [4]:
X_train_seq = X_train_w.reshape(-1, 11, 41)
X_val_seq = X_val_w.reshape(-1, 11, 41)
X_test_seq = X_test_w.reshape(-1, 11, 41)

X_train_seq = np.transpose(X_train_seq, (0, 2, 1))
X_val_seq = np.transpose(X_val_seq, (0, 2, 1))
X_test_seq = np.transpose(X_test_seq, (0, 2, 1))

print(X_train_seq.shape)

(3337092, 41, 11)


Normalization

In [5]:
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(y_train_w)
y_val_scaled = y_scaler.transform(y_val_w)
y_test_scaled = y_scaler.transform(y_test_w)

Tensors

In [6]:
import torch

X_train_tensor = torch.tensor(X_train_seq, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_seq, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_seq, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

Dataset and Loaders

In [8]:
from torch.utils.data import Dataset, DataLoader

class ProteinDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 4096

train_loader = DataLoader(ProteinDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(ProteinDataset(X_val_tensor, y_val_tensor), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(ProteinDataset(X_test_tensor, y_test_tensor), batch_size=batch_size, shuffle=False)

Stronger Multi scale CNN

In [9]:
import torch.nn as nn

class MultiScaleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.branch3 = nn.Sequential(
            nn.Conv1d(41, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        self.branch5 = nn.Sequential(
            nn.Conv1d(41, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        self.branch7 = nn.Sequential(
            nn.Conv1d(41, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        self.fuse = nn.Sequential(
            nn.Conv1d(64 * 3, 128, kernel_size=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU()
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 4)
        )

    def forward(self, x):
        b3 = self.branch3(x)
        b5 = self.branch5(x)
        b7 = self.branch7(x)

        x = torch.cat([b3, b5, b7], dim=1)
        x = self.fuse(x)
        x = self.pool(x)
        x = self.head(x)
        return x

Initialize the model 

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultiScaleCNN().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5)

Training

In [11]:
epochs = 20
best_val = float("inf")
best_state = None

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            val_loss += loss.item()

    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    scheduler.step(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        best_state = model.state_dict()

    print(f"Epoch {epoch+1}/{epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

Epoch 1/20 | Train: 0.6239 | Val: 0.4073
Epoch 2/20 | Train: 0.5577 | Val: 0.3860
Epoch 3/20 | Train: 0.4999 | Val: 0.4174
Epoch 4/20 | Train: 0.4292 | Val: 0.4059
Epoch 5/20 | Train: 0.4409 | Val: 0.4029
Epoch 6/20 | Train: 0.3696 | Val: 0.4426
Epoch 7/20 | Train: 0.2952 | Val: 0.4588
Epoch 8/20 | Train: 0.2801 | Val: 0.4605
Epoch 9/20 | Train: 0.2260 | Val: 0.4568
Epoch 10/20 | Train: 0.2079 | Val: 0.4622
Epoch 11/20 | Train: 0.1969 | Val: 0.4729
Epoch 12/20 | Train: 0.1881 | Val: 0.4782
Epoch 13/20 | Train: 0.1808 | Val: 0.4901
Epoch 14/20 | Train: 0.1749 | Val: 0.4939
Epoch 15/20 | Train: 0.1666 | Val: 0.4917
Epoch 16/20 | Train: 0.1603 | Val: 0.4952
Epoch 17/20 | Train: 0.1609 | Val: 0.4904
Epoch 18/20 | Train: 0.1547 | Val: 0.4996
Epoch 19/20 | Train: 0.1619 | Val: 0.4994
Epoch 20/20 | Train: 0.1584 | Val: 0.4984


In [12]:
model.load_state_dict(best_state)
model.eval()

MultiScaleCNN(
  (branch3): Sequential(
    (0): Conv1d(41, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (branch5): Sequential(
    (0): Conv1d(41, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (branch7): Sequential(
    (0): Conv1d(41, 64, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (fuse): Sequential(
    (0): Conv1d(192, 128, kernel_size=(1,), stride=(1,))
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(128, 256, kernel_size=(3,), stride=(1,), padding=(1,))
    (4): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
  )
  (pool): 

In [13]:
preds_scaled = []
targets_scaled = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)

        preds_scaled.append(outputs.cpu().numpy())
        targets_scaled.append(y_batch.numpy())

preds_scaled = np.vstack(preds_scaled)
targets_scaled = np.vstack(targets_scaled)

preds = y_scaler.inverse_transform(preds_scaled)
targets = y_scaler.inverse_transform(targets_scaled)

In [14]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(targets, preds))
mae = mean_absolute_error(targets, preds)
r2 = r2_score(targets, preds)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

RMSE: 757.4517723921438
MAE: 204.02911376953125
R2: 0.4288851022720337
